# Avaliação dos Resultados

Notebook responsável por avaliar a qualidade do modelo

In [43]:
import pandas as pd

from src.utils import carregar_dataset, gerar_hash_md5

from sklearn.metrics import accuracy_score, f1_score, classification_report

In [44]:
df_true = pd.read_parquet('../data/train-00000-of-00001.parquet')
df_true = df_true[(df_true.data_category_QA == 'positivo') | (df_true.data_category_QA == 'negativo')]
df_true.shape

(3290, 18)

In [45]:
df_true['md5'] = df_true.content.apply(gerar_hash_md5)

In [46]:
df_true.head()

,content,context_metadata,question,type_question,type_feature,name,occupation,instructions,chatbot_goal,adjective,data_category,id,chunks_big,classes,chosen_class_id,language,data_category_QA,content_base_uuids,md5
0,PERGUNTAS FREQUENTES\n\n ...,"{'kind': 'customsearch#result', 'title': 'Perg...",Roupas troca prazo?,simple,Lexical entropy low,Bella,Atendente de E-commerce,"[Responda sempre educadamente e com empatia, U...","Oferecer suporte sobre produtos, frete, person...",Criativo,text_generation,1,[{'content': 'PERGUNTAS FREQUENTES ...,"[{'class': 'iluminação', 'context': 'quando há...",P1,0,positivo,5db3c3c5-c9fc-44ec-991d-dc32f9bd37d5,1f98b127ca21922a8f960cc902a1ae8e
1,PERGUNTAS FREQUENTES\n\n ...,"{'kind': 'customsearch#result', 'title': 'Perg...",Qual email usar?,simple,Lexical entropy low,Bazarino,Profissão: Atendente de E-commerce,[Finalize a resposta com um emoji de animal e ...,"Facilitar trocas, prazos, contatos, dúvidas, e...",Relaxado,text_generation,2,"[{'content': 'Se, após a primeira troca, o usu...","[{'class': 'trocas_após_primeira', 'context': ...",P1,0,positivo,e7d321f2-e0f7-4ee7-aa0d-cb7947b837bf,1f98b127ca21922a8f960cc902a1ae8e
2,Perguntas Frequentes\n\nQuais as formas de pag...,"{'kind': 'customsearch#result', 'title': 'Loja...",Quais serviços usados para realizar envios?,simple,Lexical entropy low,Felicio,Atendente de E-commerce,[Responda de forma objetiva e clara sempre que...,O Chatbot é para fornecer informações sobre pa...,Generoso,text_generation,3,[{'content': 'Perguntas Frequentes Quais as f...,"[{'class': 'pagamento', 'context': 'Referente ...",P1,0,positivo,363de0bd-a2fe-4045-92d3-6af39fc97f17,e435c0c2d64653c51c92cbe286264217
3,Perguntas Frequentes\n\nQuais as formas de pag...,"{'kind': 'customsearch#result', 'title': 'Loja...",Como ser notificado de disponibilidade novamen...,simple,Lexical entropy low,Frederico.,Atendente de Suporte ao Cliente em E-commerce,[Responda sempre mencionando a Shoes Julis de ...,Proporcionar suporte completo aos clientes Sho...,Inovador,text_generation,4,[{'content': 'Para ser notificado quando o pro...,"[{'class': 'notificacao_estoque', 'context': '...",P1,0,positivo,1eb07a54-7803-4816-b146-2c8a5e82e497,e435c0c2d64653c51c92cbe286264217
4,Perguntas Frequentes\n\n1. QUAIS SÃO AS FORMAS...,"{'kind': 'customsearch#result', 'title': 'Perg...",Quais tipos cartões aceitam para realizar paga...,simple,Lexical entropy low,Maximus,Atendente de Loja Virtual,"[Em todas as respostas, enfatize a transparênc...",Auxiliar com informações sobre formas de pagam...,Sistemático,text_generation,5,[{'content': 'Perguntas Frequentes 1. QUAIS S...,"[{'class': 'política_de_privacidade', 'context...",P1,0,positivo,61f16329-e698-4106-88c7-fd23dac0a70d,d5f7d9465ecef75d8a7a04a2767fb0b0


In [47]:
df = pd.read_json('../data/gpt-5.4-nano/resultados.jsonl', lines=True)
df.shape

(6580, 6)

In [48]:
df.shape[0]/2

3290.0

In [49]:
df.head()

,tipo,md5,question,pred,contexto,model
0,rag,1f98b127ca21922a8f960cc902a1ae8e,Roupas troca prazo?,positivo,"[1f98b127ca21922a8f960cc902a1ae8e_10949, 1f98b...",gpt-5.4-nano
1,rag,1f98b127ca21922a8f960cc902a1ae8e,Qual email usar?,positivo,"[1f98b127ca21922a8f960cc902a1ae8e_10950, 1f98b...",gpt-5.4-nano
2,rag,e435c0c2d64653c51c92cbe286264217,Quais serviços usados para realizar envios?,positivo,"[e435c0c2d64653c51c92cbe286264217_10951, e435c...",gpt-5.4-nano
3,rag,e435c0c2d64653c51c92cbe286264217,Como ser notificado de disponibilidade novamen...,positivo,"[e435c0c2d64653c51c92cbe286264217_15156, e435c...",gpt-5.4-nano
4,rag,d5f7d9465ecef75d8a7a04a2767fb0b0,Quais tipos cartões aceitam para realizar paga...,positivo,"[d5f7d9465ecef75d8a7a04a2767fb0b0_10717, d5f7d...",gpt-5.4-nano


In [50]:
df = df.merge(
      df_true[['md5', 'question', 'data_category_QA', 'id', 'content']].drop_duplicates(subset=['md5', 'question']),
      on=['md5', 'question'],
      how='left'
  )

In [51]:
df.shape

(6580, 9)

In [52]:
df_true[df_true.duplicated(subset=['md5', 'question'])][['content', 'question', 'data_category_QA', 'id', 'md5']]

,content,question,data_category_QA,id,md5
1129,Perguntas frequentes\n\nGeral\n\nO que é o Mas...,Quem fundou LoungeKey?,negativo,709,cd896baffce2bad057e8a9c6fa2c68e3
1384,Onde está meu produto? Rastrear pedido 0 Carri...,"If you have received the code, how long to con...",positivo,92,269876c34663caf8cb4fd952b4664158
2426,Perguntas frequentes\n\nGeral\n\nO que é o Mas...,How many offers can I redeem daily?,negativo,710,cd896baffce2bad057e8a9c6fa2c68e3
3720,Perguntas frequentes\n\nGeral\n\nO que é o Mas...,¿Quién fundó loungekey?,negativo,709,cd896baffce2bad057e8a9c6fa2c68e3


In [53]:
df_true[df_true.duplicated(subset=['md5', 'question'], keep=False)]\
      .sort_values(['md5', 'question'])[['content', 'question', 'data_category_QA', 'id', 'md5']]

,content,question,data_category_QA,id,md5
1383,Onde está meu produto? Rastrear pedido 0 Carri...,"If you have received the code, how long to con...",positivo,91,269876c34663caf8cb4fd952b4664158
1384,Onde está meu produto? Rastrear pedido 0 Carri...,"If you have received the code, how long to con...",positivo,92,269876c34663caf8cb4fd952b4664158
2425,Perguntas frequentes\n\nGeral\n\nO que é o Mas...,How many offers can I redeem daily?,negativo,709,cd896baffce2bad057e8a9c6fa2c68e3
2426,Perguntas frequentes\n\nGeral\n\nO que é o Mas...,How many offers can I redeem daily?,negativo,710,cd896baffce2bad057e8a9c6fa2c68e3
791,Perguntas frequentes\n\nGeral\n\nO que é o Mas...,Quem fundou LoungeKey?,negativo,1098,cd896baffce2bad057e8a9c6fa2c68e3
1129,Perguntas frequentes\n\nGeral\n\nO que é o Mas...,Quem fundou LoungeKey?,negativo,709,cd896baffce2bad057e8a9c6fa2c68e3
3382,Perguntas frequentes\n\nGeral\n\nO que é o Mas...,¿Quién fundó loungekey?,negativo,1098,cd896baffce2bad057e8a9c6fa2c68e3
3720,Perguntas frequentes\n\nGeral\n\nO que é o Mas...,¿Quién fundó loungekey?,negativo,709,cd896baffce2bad057e8a9c6fa2c68e3


In [54]:
df_rag = df[df.tipo == 'rag']
df_rag = df_rag[~df_rag.duplicated(subset=['md5', 'question'])]
df_rag.shape

(3286, 9)

In [55]:
df_full_context = df[df.tipo == 'full_context']
df_full_context = df_full_context[~df_full_context.duplicated(subset=['md5', 'question'])]
df_full_context.shape

(3286, 9)

In [56]:
print(classification_report(df_rag.data_category_QA, df_rag.pred))

              precision    recall  f1-score   support

    negativo       0.70      0.97      0.81      1566
    positivo       0.96      0.62      0.75      1720

    accuracy                           0.79      3286
   macro avg       0.83      0.79      0.78      3286
weighted avg       0.83      0.79      0.78      3286



In [57]:
print(classification_report(df_full_context.data_category_QA, df_full_context.pred))

              precision    recall  f1-score   support

    negativo       0.75      0.96      0.84      1566
    positivo       0.95      0.71      0.82      1720

    accuracy                           0.83      3286
   macro avg       0.85      0.84      0.83      3286
weighted avg       0.86      0.83      0.83      3286



In [58]:
df_full_context.head()

,tipo,md5,question,pred,contexto,model,data_category_QA,id,content
3290,full_context,1f98b127ca21922a8f960cc902a1ae8e,Roupas troca prazo?,positivo,None,gpt-5.4-nano,positivo,1,PERGUNTAS FREQUENTES\n\n ...
3291,full_context,1f98b127ca21922a8f960cc902a1ae8e,Qual email usar?,positivo,None,gpt-5.4-nano,positivo,2,PERGUNTAS FREQUENTES\n\n ...
3292,full_context,e435c0c2d64653c51c92cbe286264217,Quais serviços usados para realizar envios?,positivo,None,gpt-5.4-nano,positivo,3,Perguntas Frequentes\n\nQuais as formas de pag...
3293,full_context,e435c0c2d64653c51c92cbe286264217,Como ser notificado de disponibilidade novamen...,positivo,None,gpt-5.4-nano,positivo,4,Perguntas Frequentes\n\nQuais as formas de pag...
3294,full_context,d5f7d9465ecef75d8a7a04a2767fb0b0,Quais tipos cartões aceitam para realizar paga...,positivo,None,gpt-5.4-nano,positivo,5,Perguntas Frequentes\n\n1. QUAIS SÃO AS FORMAS...


In [59]:
df_full_context.columns

Index(['tipo', 'md5', 'question', 'pred', 'contexto', 'model',
       'data_category_QA', 'id', 'content'],
      dtype='object')

In [60]:
df_full_context.to_csv('../data/gpt-5.4-nano/results.csv', index=False)